In [1]:
import festim as F
from foam2dolfinx import OpenFOAMReader, find_closest_value
from dolfinx.io import gmsh
from mpi4py import MPI
import numpy as np

# instantiate reader:

/Users/ckhurana/miniconda3/envs/tes-pav-env/lib/python3.10/site-packages/festim/coupled_heat_hydrogen_problem.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  import tqdm.autonotebook


In [2]:
## Reading mesh

mesh_path = "/Users/ckhurana/FESTIM/FESTIM-dev/openfoam/OpenFOAM/shellTubeHX/Fusion-Heat-Exchangers/openfoam/cylinder_with_interfaces/partitioned_cylinder_with_festim_interfaces.msh"

mesh_data = gmsh.read_from_msh(
    mesh_path, MPI.COMM_WORLD, 0, gdim=3
)

mesh = mesh_data.mesh

ft = mesh_data.facet_tags
ct = mesh_data.cell_tags

from dolfinx import plot
import pyvista

fdim = mesh.topology.dim - 1
tdim = mesh.topology.dim
mesh.topology.create_connectivity(fdim, tdim)
topology, cell_types, x = plot.vtk_mesh(mesh, fdim, ft.indices)

p = pyvista.Plotter()
grid = pyvista.UnstructuredGrid(topology, cell_types, x)
grid.cell_data["Facet Marker"] = ft.values
grid.set_active_scalars("Facet Marker")
p.add_mesh(grid, show_edges=True)
if pyvista.OFF_SCREEN:
    figure = p.screenshot("facet_marker.png")
p.show()

Info    : Reading '/Users/ckhurana/FESTIM/FESTIM-dev/openfoam/OpenFOAM/shellTubeHX/Fusion-Heat-Exchangers/openfoam/cylinder_with_interfaces/partitioned_cylinder_with_festim_interfaces.msh'...
Info    : 35141 nodes
Info    : 222956 elements
Info    : Done reading '/Users/ckhurana/FESTIM/FESTIM-dev/openfoam/OpenFOAM/shellTubeHX/Fusion-Heat-Exchangers/openfoam/cylinder_with_interfaces/partitioned_cylinder_with_festim_interfaces.msh'


Widget(value='<iframe src="http://localhost:62464/index.html?ui=P_0x103d19330_0&reconnect=auto" class="pyvista…

In [3]:
my_reader = OpenFOAMReader(filename="/Users/ckhurana/FESTIM/FESTIM-dev/openfoam/OpenFOAM/shellTubeHX/Fusion-Heat-Exchangers/openfoam/cylinder_with_interfaces/case.foam", cell_type=10)

def get_my_U_field(t, region="top_fluid"):
    closest_t = find_closest_value(my_reader.reader.time_values, float(t))
    print("reading field at time:", closest_t, "region:", region)
    u_field = my_reader.create_dolfinx_function(t=closest_t, name="U", subdomain=region)
    return u_field

def get_my_T_field(t, region="top_fluid"):
    closest_t = find_closest_value(my_reader.reader.time_values, float(t))
    print("reading field at time:", closest_t, "region:", region)
    T_field = my_reader.create_dolfinx_function(t=closest_t, name="T", subdomain=region)
    return T_field


print(my_reader.reader.time_values)

[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 12.5, 13.0, 13.5, 14.0, 14.5, 15.0, 15.5, 16.0, 16.5, 17.0, 17.5, 18.0, 18.5, 19.0, 19.5, 20.0, 20.5, 21.0, 21.5, 22.0, 22.5, 23.0, 23.5, 24.0, 24.5, 25.0, 25.5, 26.0, 26.5, 27.0, 27.5, 28.0, 28.5, 29.0, 29.5, 30.0, 30.5, 31.0, 31.5, 32.0, 32.5, 33.0, 33.5, 34.0, 34.5, 35.0, 35.5, 36.0, 36.5, 37.0, 37.5, 38.0, 38.5, 39.0, 39.5, 40.0, 40.5, 41.0, 41.5, 42.0, 42.5, 43.0, 43.5, 44.0, 44.5, 45.0, 45.5, 46.0, 46.5, 47.0, 47.5, 48.0, 48.5, 49.0, 49.5, 50.0]


In [6]:
top_fluid = F.Material(D_0=1e-3, E_D=0, K_S_0=10, E_K_S=0)
bottom_fluid = F.Material(D_0=1e-3, E_D=0, K_S_0=10, E_K_S=0)
slab = F.Material(D_0=1e-4, E_D=0, K_S_0=5, E_K_S=0)

top_vol = F.VolumeSubdomain(id=1, material=top_fluid)
bottom_vol = F.VolumeSubdomain(id=3, material=bottom_fluid)
slab_vol = F.VolumeSubdomain(id=2, material=slab)

walls = F.SurfaceSubdomain(id=99)
top_inlet = F.SurfaceSubdomain(id=4)
top_outlet = F.SurfaceSubdomain(id=2)
bottom_inlet = F.SurfaceSubdomain(id=14)
bottom_outlet = F.SurfaceSubdomain(id=11)
top_slab_interface = F.SurfaceSubdomain(id=3)
bottom_slab_interface = F.SurfaceSubdomain(id=13)

my_model = F.HydrogenTransportProblemDiscontinuous()

my_model.mesh = F.Mesh(mesh)

# we need to pass the meshtags to the model directly
my_model.facet_meshtags = ft
my_model.volume_meshtags = ct

my_model.subdomains = [top_inlet, top_outlet, bottom_inlet, bottom_outlet, walls, top_vol, bottom_vol, slab_vol]

my_model.surface_to_volume = {
    top_inlet: top_vol,
    top_outlet: top_vol,
    bottom_inlet: bottom_vol,
    bottom_outlet: bottom_vol,
    top_slab_interface: top_vol,
    bottom_slab_interface: slab_vol,
}


my_model.interfaces = [
    F.Interface(id=3, subdomains=[top_vol, slab_vol], penalty_term=1e10),
    F.Interface(id=4, subdomains=[slab_vol, bottom_vol], penalty_term=1e10),
]

H = F.Species("H", subdomains=[top_vol, bottom_vol, slab_vol])
my_model.species = [H]

my_model.temperature = lambda t: get_my_T_field(t)
# my_model.temperature = 400

my_model.boundary_conditions = [
    # F.FixedConcentrationBC(subdomain=outlet, value=0, species=H),
    # F.FixedConcentrationBC(subdomain=top_surface, value=1, species=H),
    F.FixedConcentrationBC(subdomain=top_inlet, value=1, species=H),
    # F.FixedConcentrationBC(subdomain=bottom_inlet, value=2, species=H),
    # F.SievertsBC(subdomain=top_slab_interface, S_0=slab.K_S_0, E_S=slab.E_K_S, pressure=100, species=H),
    # F.SievertsBC(subdomain=bottom_slab_interface, S_0=slab.K_S_0, E_S=slab.E_K_S, pressure=100, species=H)
]

advection_term_top = F.AdvectionTerm(
    velocity=lambda t: get_my_U_field(t, region="top_fluid"),
    subdomain=top_vol,
    species=H,
)

advection_term_bottom = F.AdvectionTerm(
    velocity=lambda t: get_my_U_field(t, region="bottom_fluid"),
    subdomain=bottom_vol,
    species=H,
)


my_model.exports = [
    F.VTXSpeciesExport(filename="top.bp", field=H, subdomain=top_vol),
    F.VTXSpeciesExport(filename="bottom.bp", field=H, subdomain=bottom_vol),
    F.VTXSpeciesExport(filename="slab.bp", field=H, subdomain=slab_vol),
]

my_model.advection_terms = [advection_term_top, advection_term_bottom]
my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, stepsize=0.5, final_time=50)

my_model.initialise()
my_model.run() 

reading field at time: 0.0 region: top_fluid
reading field at time: 0.0 region: top_fluid


ValueError: self.temperature should return a float or an int, not <class 'dolfinx.fem.function.Function'> 